# Task 3: Machine Learning – Student Segmentation & Risk Detection

## 🚀 Objective
In this task, we use unsupervised machine learning (K-Means Clustering) to discover hidden student groups and behavioral patterns. Unlike rule-based scoring (SRI), clustering groups students based on multidimensional similarities across academics, wellness, and engagement.

In [ ]:
from pathlib import Path
import os

# Robust Path Handling for Notebooks
BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Data Preprocessing & Feature Selection

### 📝 Feature Selection Justification
To build a comprehensive student behavioral profile, I have selected features across four key dimensions:
1. **Academics**: `gpa`, `attendance`, `assignments_completion` — Indicate basic performance and consistency.
2. **Wellness**: `stress_level`, `sleep_hours`, `mental_wellbeing` — Captured to identify burnout risks.
3. **Efficiency**: `productivity_score`, `distractions` — Show how effectively a student uses their time.
4. **Aspiration**: `career_clarity`, `skill_readiness`, `engagement_score` — Measure motivation and future readiness.

**Why these?** These features directly influence student success. By clustering them, we can find students who might have high GPAs but high stress, or students with potential but low clarity.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# Load the scored dataset
df = pd.read_csv(DATA_DIR / 'students_scored.csv')

# Select relevant numerical features
features = [
    'gpa', 'attendance', 'assignments_completion', 
    'stress_level', 'sleep_hours', 'mental_wellbeing', 
    'productivity_score', 'distractions', 
    'career_clarity', 'skill_readiness', 'engagement_score'
]
X = df[features]

print(f"Selected {len(features)} features for clustering.")

### ⚖️ Data Scaling (Standardization)
K-Means is distance-based, so features with larger scales (like engagement_score 0-100) would dominate those with smaller scales (like gpa 0-10). 

**Step**: I apply `StandardScaler` to normalize all features to have a mean of 0 and a standard deviation of 1.

In [ ]:
# Applying StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data standardization complete.")

## 2. Determining Optimal K (Elbow Method & Silhouette Score)
To find the best number of clusters (K), I calculate the Within-Cluster Sum of Squares (WCSS). We look for the 'elbow' point where the rate of decrease in WCSS slows down significantly.

In [ ]:
from sklearn.metrics import silhouette_score

wcss = []
silhouette_avg = []

for i in range(2, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_avg.append(silhouette_score(X_scaled, kmeans.labels_))

plt.figure(figsize=(10, 5))
plt.plot(range(2, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

### 📝 Choice of K Justification
Observing the Elbow plot, the drop in WCSS is steepest until **K=3** and **K=4**, after which it flattens out. 

**Decision**: I choose **K=4**. 
**Why?** Having 4 clusters allows us to differentiate between 'Average' students and truly 'Disengaged' students, as well as separating 'High Achievers' from 'Balanced Students'. K=4 provides the most actionable segmentation for mentors without over-complicating the model.

## 3. Applying K-Means (K=4)

In [ ]:
k = 4
kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Analysis of Cluster Centroids to interpret behavior
cluster_analysis = df.groupby('cluster')[features + ['SRI']].mean().round(2)
print("Cluster Profile Centroids:")
cluster_analysis

## 4. Visualization (PCA and Cluster Plots)
Since we have 11 features, we cannot plot them in 2D. I use **Principal Component Analysis (PCA)** to project the data into two principal components (dimensions) that capture the maximum variance.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['pca1'] = X_pca[:, 0]
df['pca2'] = X_pca[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(x='pca1', y='pca2', hue='cluster', data=df, palette='Set1', s=100, alpha=0.8)
plt.title('High-Level Cluster Segmentation (2D PCA)')
plt.xlabel('Principal Component 1 (General Performance/Engagement)')
plt.ylabel('Principal Component 2 (Wellness/Stress Factors)')
plt.legend(title='Cluster ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### 📝 Visualization Interpretation
The PCA plot shows distinct grouping. principal component 1 generally separates students by overall performance (GPA/Attendance), while Component 2 often separates them based on Wellness/Stress dynamics. 
- **Cluster 0 (Red)**: Students at extreme ends of the stress spectrum with low performance.
- **Cluster 2 (Green)**: Stable, high-performing students clustered together.
- **Cluster 1 & 3**: Intermediate students with mixed clarity and engagement levels.

## 5. Comparison: ML Clusters vs Rule-Based Intelligence (SRI)
We compare the discovered clusters with the SRI categories to see if ML reveals risks that the simple math formulas might overlook.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='cluster', y='SRI', data=df, palette='Set2')
plt.title('How Rules (SRI) Compare to ML Clusters')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("Agreement vs Contradiction Table:")
pd.crosstab(df['cluster'], df['category'])